# VLM Input Pruning Colab Demo

이 노트북은 고해상도 이미지를 `scene context text + selected high-res crop canvas` 형태로 바꾸는 전처리 파이프라인을 실행합니다.

기본값은 mock mode라서 무거운 VLM/YOLO 모델 없이 끝까지 실행됩니다. YOLO를 켜면 Colab에서 `ultralytics`를 추가 설치합니다.

In [ ]:
# 1. GitHub repo clone
# 이미 /content/vlm_input_pruning 폴더가 있으면 삭제 후 다시 clone합니다.
import os
from pathlib import Path

!rm -rf /content/vlm_input_pruning
!git clone https://github.com/tjdxo/vlm_input_pruning.git /content/vlm_input_pruning

os.chdir("/content/vlm_input_pruning")
print("cwd:", Path.cwd())

In [ ]:
# 2. 기본 의존성 설치
!pip install -q -r requirements.txt

In [ ]:
# 3. 입력 이미지 선택
# 기본 샘플 입력은 repo에 포함되어 있습니다.
from pathlib import Path

USE_SAMPLE_IMAGE = True
sample_image = Path("examples/sample_inputs/test.jpg")

if USE_SAMPLE_IMAGE:
    image_path = sample_image
else:
    from google.colab import files
    uploaded = files.upload()
    image_path = Path(next(iter(uploaded.keys())))

question = "Describe the important objects while preserving fine details."
out_dir = Path("/content/outputs")
print("image_path:", image_path)
print("question:", question)
print("out_dir:", out_dir)

In [ ]:
# 4. 원본 이미지 확인
from PIL import Image
from IPython.display import display

img = Image.open(image_path).convert("RGB")
print("original size:", img.size)
display(img.resize((min(900, img.width), int(img.height * min(900, img.width) / img.width))))

In [ ]:
# 5. Mock mode end-to-end 실행
# small_vlm=mock, detector=dummy가 기본값입니다.
from src.config import load_config
from src.pipeline import run_pipeline

config = load_config("configs/default.yaml")
config["small_vlm"]["backend"] = "mock"
config["detector"]["backend"] = "dummy"

result = run_pipeline(
    image_path=image_path,
    question=question,
    config=config,
    out_dir=out_dir,
    call_main_vlm=False,
)

print("metadata:", result["metadata_path"])
print("composed image:", result["composed_image_path"])
print("detections visualization:", result["detections_visualization_path"])
print("final prompt:", result["final_prompt_path"])

In [ ]:
# 6. Small VLM stage 출력 확인
print(result["scene_context"]["text"])
print("\nimportance hints:")
for hint in result["scene_context"].get("importance_hints", []):
    print("-", hint)

In [ ]:
# 7. Detection 결과 확인
import pandas as pd
from IPython.display import display

display(pd.DataFrame(result["detections"]))
display(Image.open(result["detections_visualization_path"]))

In [ ]:
# 8. 선택된 crop metadata 확인
display(pd.DataFrame(result["crop_metadata"]))

In [ ]:
# 9. 개별 crop 이미지 확인
from pathlib import Path

crop_paths = [Path(p) for p in result.get("crop_image_paths", [])]
print("num crops:", len(crop_paths))
for path in crop_paths:
    print(path.name)
    display(Image.open(path))

In [ ]:
# 10. 최종 composed crop canvas 확인
composed = Image.open(result["composed_image_path"])
print("composed size:", composed.size)
display(composed)

In [ ]:
# 11. Token estimate / latency 확인
import json

print(json.dumps(result["metrics"], indent=2, ensure_ascii=False))

In [ ]:
# 12. Main VLM에 넘길 final prompt 확인
prompt = Path(result["final_prompt_path"]).read_text(encoding="utf-8")
print(prompt)

## Optional: YOLO detector 사용

아래 셀은 Ultralytics YOLO를 설치하고 detector backend를 `yolo`로 바꿉니다. 무료 Colab에서도 `yolov8n.pt` 기준으로 가볍게 테스트할 수 있습니다.

In [ ]:
# 13. Optional YOLO 실행
# 필요할 때만 실행하세요.
!pip install -q ultralytics

config = load_config("configs/default.yaml")
config["small_vlm"]["backend"] = "mock"
config["detector"]["backend"] = "yolo"
config["detector"]["yolo_model"] = "yolov8n.pt"

yolo_result = run_pipeline(
    image_path=image_path,
    question=question,
    config=config,
    out_dir=Path("/content/outputs_yolo"),
    call_main_vlm=False,
)

print(json.dumps(yolo_result["metrics"], indent=2, ensure_ascii=False))
display(Image.open(yolo_result["detections_visualization_path"]))
display(Image.open(yolo_result["composed_image_path"]))

## Optional: 결과 다운로드

Colab 세션의 `/content/outputs` 폴더를 zip으로 묶어 다운로드합니다.

In [ ]:
!cd /content && zip -qr outputs.zip outputs
from google.colab import files
files.download('/content/outputs.zip')